# Gold Layer - Dimensional Model for Power BI

Builds the star schema that powers the dashboard: customer, product,
seller, and date dimensions, plus order-grain and item-grain fact
tables. Kept as two separate facts (not one) so revenue never gets
double-counted when slicing by product vs by order.

**Input:** `workspace.silver.*`
**Output:** `workspace.gold.*`

## Setup

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

SILVER = "workspace.silver"
GOLD = "workspace.gold"


def read_silver(table_name: str) -> DataFrame:
    return spark.table(f"{SILVER}.{table_name}")


def write_gold(df: DataFrame, table_name: str) -> None:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.{table_name}")
    print(f"wrote {table_name}: {df.count()} rows")

## dim_date

One row per calendar day, spanning the full order history. This is
what drives the All time / 90 days / 30 days / 7 days toggle on the
revenue chart - Power BI filters this table by relative date instead
of us hardcoding date logic anywhere.

In [0]:
orders_silver = read_silver("orders")

date_bounds = orders_silver.select(
    F.min("order_purchase_timestamp").alias("min_date"),
    F.max("order_purchase_timestamp").alias("max_date"),
).collect()[0]

dim_date = (
    spark.sql(
        f"""
        SELECT explode(sequence(
            to_date('{date_bounds['min_date']}'),
            to_date('{date_bounds['max_date']}'),
            interval 1 day
        )) AS date_key
        """
    )
    .withColumn("year", F.year("date_key"))
    .withColumn("month", F.month("date_key"))
    .withColumn("month_name", F.date_format("date_key", "MMMM"))
    .withColumn("quarter", F.quarter("date_key"))
    .withColumn("day_of_week", F.date_format("date_key", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("date_key").isin([1, 7]))
)

write_gold(dim_date, "dim_date")

## dim_customer

One row per unique customer (not per customer_id - Olist gives every
order a fresh customer_id, so customer_unique_id is the real person).
Order count here is what drives the new-vs-returning split on the
Home page donut.

In [0]:
customers_silver = read_silver("customers")

customer_order_counts = (
    orders_silver
    .join(customers_silver, on="customer_id", how="inner")
    .groupBy("customer_unique_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.min("order_purchase_timestamp").alias("first_order_date"),
        F.max("order_purchase_timestamp").alias("last_order_date"),
    )
)

dim_customer = (
    customers_silver
    .dropDuplicates(["customer_unique_id"])
    .select("customer_unique_id", "customer_city", "customer_state", "customer_zip_code_prefix")
    .join(customer_order_counts, on="customer_unique_id", how="left")
    .withColumn("is_returning_customer", F.col("total_orders") > 1)
)

write_gold(dim_customer, "dim_customer")

## dim_product

Enriched with two things the raw data doesn't have: simulated stock,
and pre-aggregated sales stats so the "top products" table on the
Home page doesn't need a heavy live join in Power BI.

Stock is simulated but deterministic - hashing product_id into a
0-500 range means the same product always gets the same fake stock
number every time this runs, instead of a random value that changes
on every refresh.

In [0]:
products_silver = read_silver("products")
order_items_silver = read_silver("order_items")

product_sales_stats = (
    order_items_silver
    .join(orders_silver.select("order_id", "order_purchase_timestamp"), on="order_id")
    .groupBy("product_id")
    .agg(
        F.sum("price").alias("total_revenue"),
        F.count("order_item_id").alias("total_units_sold"),
    )
)

dim_product = (
    products_silver
    .withColumn("simulated_stock", (F.abs(F.hash("product_id")) % 500) + 10)
    .join(product_sales_stats, on="product_id", how="left")
    .withColumn("total_revenue", F.coalesce(F.col("total_revenue"), F.lit(0)))
    .withColumn("total_units_sold", F.coalesce(F.col("total_units_sold"), F.lit(0)))
)

write_gold(dim_product, "dim_product")

## dim_seller

In [0]:
dim_seller = read_silver("sellers")

write_gold(dim_seller, "dim_seller")

## fact_orders

One row per order - this is what the Home page KPIs and the whole
Orders list page run on. Payments get collapsed to one row per order
here (primary method = the first payment_sequential), since an order
can't show five payment methods in a single list row.

In [0]:
order_payments_silver = read_silver("order_payments")
order_reviews_silver = read_silver("order_reviews")

payment_window = Window.partitionBy("order_id").orderBy("payment_sequential")

primary_payment = (
    order_payments_silver
    .withColumn("row_num", F.row_number().over(payment_window))
    .filter(F.col("row_num") == 1)
    .select(
        "order_id",
        F.col("payment_type").alias("primary_payment_type"),
        F.col("payment_installments").alias("primary_payment_installments"),
    )
)

payment_totals = (
    order_payments_silver
    .groupBy("order_id")
    .agg(F.sum("payment_value").alias("total_payment_value"))
)

order_item_totals = (
    order_items_silver
    .groupBy("order_id")
    .agg(
        F.sum("price").alias("order_revenue"),
        F.sum("freight_value").alias("order_freight"),
        F.count("order_item_id").alias("item_count"),
    )
)

review_scores = order_reviews_silver.select("order_id", "review_score")

fact_orders = (
    orders_silver
    .join(customers_silver.select("customer_id", "customer_unique_id"), on="customer_id", how="left")
    .join(order_item_totals, on="order_id", how="left")
    .join(primary_payment, on="order_id", how="left")
    .join(payment_totals, on="order_id", how="left")
    .join(review_scores, on="order_id", how="left")
    .select(
        "order_id",
        "customer_unique_id",
        "order_status",
        F.to_date("order_purchase_timestamp").alias("order_date"),
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "is_on_time",
        "order_revenue",
        "order_freight",
        "item_count",
        "primary_payment_type",
        "primary_payment_installments",
        "total_payment_value",
        "review_score",
    )
)

write_gold(fact_orders, "fact_orders")

## fact_order_items

Item-grain fact - this is what the top products table and the
product-category revenue donut run on, and what populates the order
items block on the Orders detail page. Category is denormalized in
here directly so Power BI doesn't need to hop through dim_product
just to slice revenue by category.

In [0]:
fact_order_items = (
    order_items_silver
    .join(orders_silver.select("order_id", "order_purchase_timestamp"), on="order_id")
    .join(
        products_silver.select("product_id", "product_category_name_english"),
        on="product_id",
        how="left",
    )
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "product_category_name_english",
        "seller_id",
        F.to_date("order_purchase_timestamp").alias("order_date"),
        "price",
        "freight_value",
    )
)

write_gold(fact_order_items, "fact_order_items")

## Sanity check

In [0]:
display(spark.sql("SHOW TABLES IN workspace.gold"))

In [0]:
# revenue by category - segment donut for the Home page
display(
    spark.sql("""
        SELECT product_category_name_english, SUM(price) AS revenue
        FROM workspace.gold.fact_order_items
        GROUP BY product_category_name_english
        ORDER BY revenue DESC
        LIMIT 10
    """)
)